
# M2.851 · Tipología y ciclo de vida de los datos
## Práctica 1 · Dataset de tecnologías utilizadas en proyectos de hardware abierto certificado

En esta práctica se utilizará como fuente el directorio de proyectos certificados de la **Open Source Hardware Association (OSHWA)**:

https://certification.oshwa.org/list.html

OSHWA mantiene un programa de certificación de hardware abierto. Para que un proyecto pueda certificarse debe publicar la documentación necesaria para que otras personas puedan estudiar, modificar y reproducir el hardware. Esta documentación puede incluir los ficheros de diseño, esquemas y la lista de materiales o **Bill of Materials (BOM)**.

El objetivo de esta práctica no será únicamente obtener un listado de proyectos. Se pretende construir un dataset que permita estudiar **qué tecnologías aparecen en proyectos reales de hardware abierto** y que pueda servir como base para trabajos posteriores de recomendación o selección de tecnologías y componentes.

La idea general será:

**Directorio OSHWA → descubrir proyectos → entrar en cada ficha → extraer metadatos → detectar tecnologías mencionadas → construir dataset**

Ejemplos de preguntas que este dataset podría permitir estudiar posteriormente:

- ¿Qué microcontroladores aparecen con más frecuencia en proyectos de hardware abierto?
- ¿Qué tecnologías de comunicación suelen aparecer juntas?
- ¿Qué tecnologías aparecen en proyectos de electrónica, robótica o IoT?
- ¿Cómo cambia el uso de determinadas tecnologías con el tiempo?
- ¿Qué proyectos podrían utilizarse como referencia para recomendar tecnologías a partir de una descripción de requisitos?

La práctica actual se centra en la **captura mediante web scraping**. No se entrenará todavía ningún modelo de machine learning.

Tampoco se utilizará la API oficial de OSHWA como fuente principal, ya que el enunciado de la práctica indica que una API no puede sustituir al proceso de web scraping.

### Fuentes utilizadas para justificar la elección de OSHWA

- OSHWA - About:  
  https://oshwa.org/about/

- Open Source Hardware Basics:  
  https://certification.oshwa.org/basics.html

- OSHWA Certification - Documentation:  
  https://certification.oshwa.org/process/documentation.html

- OSHWA Certification FAQ:  
  https://certification.oshwa.org/basics/faq.html



## Carga de librerías

Empezamos cargando las librerías que necesitaremos para realizar el scraping:

- `requests`: realizar peticiones HTTP.
- `BeautifulSoup`: analizar el contenido HTML.
- `pandas`: organizar los registros obtenidos en un DataFrame.
- `urllib.robotparser`: interpretar el archivo `robots.txt`.
- `urljoin`: convertir enlaces relativos en enlaces completos.
- `re`: localizar patrones en URLs y textos.
- `time` y `random`: espaciar las peticiones para evitar realizar muchas solicitudes consecutivas.


In [1]:
import re
import time
import random
import requests
import pandas as pd
import bs4

from bs4 import BeautifulSoup
from urllib.parse import urljoin, unquote
from urllib.robotparser import RobotFileParser

print("Librerías importadas")
print("requests:", requests.__version__)
print("BeautifulSoup:", bs4.__version__)
print("pandas:", pd.__version__)

Librerías importadas
requests: 2.34.2
BeautifulSoup: 4.15.0
pandas: 3.0.5



## Configuración inicial

Para las primeras pruebas se mantiene un número reducido de proyectos. Primero se comprobará que el programa encuentra correctamente las fichas y extrae sus campos.

En esta versión seguimos empezando con proyectos cuyo tipo principal sea **Electronics**, porque son los más relacionados con el objetivo de estudiar tecnologías electrónicas.

Una vez validado el scraper se podrá:

- aumentar `MAX_PROYECTOS`;
- asignar `MAX_PROYECTOS = None` para procesar todos los proyectos seleccionados;
- asignar `SOLO_ELECTRONICA = False` para incorporar otros tipos como Robotics, Science, Education, Tool o Environmental.

De este modo el mismo programa puede generar desde una muestra pequeña de prueba hasta un dataset mucho mayor.


In [2]:
BASE_URL = "https://certification.oshwa.org"
URL_DIRECTORIO = BASE_URL + "/list.html"
ROBOTS_URL = BASE_URL + "/robots.txt"

# User-Agent identificable. No intentamos hacernos pasar por un navegador.
USER_AGENT = "M2.851-UOC-Practica1/1.0 (academic web scraping)"

TIMEOUT = 30

# Espera entre peticiones a fichas individuales.
ESPERA_MIN = 1.5
ESPERA_MAX = 3.0

# Primera prueba
SOLO_ELECTRONICA = True
MAX_PROYECTOS = 20

print("Directorio:", URL_DIRECTORIO)
print("User-Agent:", USER_AGENT)
print("Máximo de proyectos en esta prueba:", MAX_PROYECTOS)

Directorio: https://certification.oshwa.org/list.html
User-Agent: M2.851-UOC-Practica1/1.0 (academic web scraping)
Máximo de proyectos en esta prueba: 20



## 1. Comprobación de `robots.txt`

Antes de realizar el scraping se intenta consultar `robots.txt`.

En lugar de utilizar directamente `RobotFileParser.read()`, descargaremos el archivo con `requests` y, si existe, pasaremos las líneas al método `parse()`. Esto permite utilizar el mismo `User-Agent` definido para la práctica y controlar los posibles códigos HTTP.

**Bibliografía utilizada para esta celda:**

- Stack Overflow. Ejemplo prácticamente idéntico: descargar `robots.txt` manualmente con una cabecera `User-Agent` y posteriormente utilizar `RobotFileParser.parse()` y `can_fetch()`:  
  https://stackoverflow.com/questions/37934316/change-user-agent-used-with-robotparser-in-python

- Documentación oficial de Python sobre `urllib.robotparser`:  
  https://docs.python.org/3/library/urllib.robotparser.html

- Material de la asignatura: *Web scraping*, apartado de evaluación inicial y `robots.txt`.


In [3]:
# Comprobamos robots.txt antes de descargar las fichas.
#
# Bibliografía:
#              - https://stackoverflow.com/questions/37934316/change-user-agent-used-with-robotparser-in-python
#              - https://docs.python.org/3/library/urllib.robotparser.html
#
# El ejemplo de Stack Overflow descarga robots.txt por separado y después utiliza parse(),
# que es el mismo procedimiento que adapto aquí.

robot_parser = None
robots_estado = None

try:
    respuesta_robots = requests.get(
        ROBOTS_URL,
        headers={"User-Agent": USER_AGENT},
        timeout=TIMEOUT
    )

    robots_estado = respuesta_robots.status_code
    print("Código HTTP de robots.txt:", robots_estado)

    if robots_estado == 200:
        robot_parser = RobotFileParser()
        robot_parser.set_url(ROBOTS_URL)
        robot_parser.parse(respuesta_robots.text.splitlines())

        print("robots.txt encontrado")
        print(
            "¿Permite acceder al directorio?:",
            robot_parser.can_fetch(USER_AGENT, URL_DIRECTORIO)
        )

    elif robots_estado == 404:
        print("No existe un archivo robots.txt en esta ruta (HTTP 404).")

    else:
        print(
            "No se ha podido comprobar robots.txt correctamente.",
            "Por seguridad no se continuará automáticamente hasta revisar el resultado."
        )

except requests.RequestException as error:
    print("Error al consultar robots.txt:", error)

Código HTTP de robots.txt: 200
robots.txt encontrado
¿Permite acceder al directorio?: True



La celda anterior no intenta evitar ninguna restricción. Si el servidor devuelve un `robots.txt` válido se comprobará `can_fetch()` antes de realizar las peticiones.

Si devuelve `404`, simplemente significa que no existe un archivo `robots.txt` en esa ruta. Si devuelve otro error, se detendrá el scraping automático para revisar el motivo antes de continuar.



## 2. Crear una sesión y una función de descarga

Utilizaremos `requests.Session()` para reutilizar las mismas cabeceras y configuración en todas las peticiones.

La función:

1. comprueba `robots.txt` cuando se ha podido cargar;
2. introduce una espera aleatoria;
3. realiza la petición con `timeout`;
4. comprueba el código HTTP;
5. detiene el proceso si recibe `403` o `429`.

**Bibliografía utilizada para esta celda:**

- Stack Overflow. Ejemplo de scraping que crea `requests.Session()`, realiza una petición inicial, obtiene enlaces y después reutiliza la sesión para acceder a cada subpágina:  
  https://stackoverflow.com/questions/33135929/accessing-tabular-data-via-hyperlinks-with-beautifulsoup

- GeeksForGeeks. Ejemplo de scraping de múltiples páginas donde se introduce `sleep()` entre peticiones:  
  https://www.geeksforgeeks.org/python/how-to-scrape-multiple-pages-of-a-website-using-python/

- GeeksForGeeks. Otro ejemplo que utiliza `requests.get(..., headers=...)` y `time.sleep()` para evitar peticiones demasiado rápidas:  
  https://www.geeksforgeeks.org/python/web-scraping-for-stock-prices-in-python/

- Requests, documentación oficial sobre sesiones:  
  https://requests.readthedocs.io/en/latest/user/advanced/


In [4]:
# Creamos una sesión para utilizar las mismas cabeceras durante todo el proceso.
#
# Bibliografía:
#              - https://stackoverflow.com/questions/33135929/accessing-tabular-data-via-hyperlinks-with-beautifulsoup
#              - https://www.geeksforgeeks.org/python/how-to-scrape-multiple-pages-of-a-website-using-python/
#              - https://www.geeksforgeeks.org/python/web-scraping-for-stock-prices-in-python/
#              - https://requests.readthedocs.io/en/latest/user/advanced/

sesion = requests.Session()
sesion.headers.update({
    "User-Agent": USER_AGENT,
    "Accept-Language": "en-US,en;q=0.9"
})


def descargar_pagina(url, esperar=True):
    """Descarga una página respetando robots.txt cuando esté disponible."""

    # Si robots.txt existe y prohíbe la URL, no hacemos la petición.
    if robot_parser is not None:
        if not robot_parser.can_fetch(USER_AGENT, url):
            raise PermissionError("robots.txt no permite acceder a: " + url)

    # Si robots.txt ha devuelto un error distinto de 200/404 prefiero detenerme.
    if robots_estado not in (None, 200, 404):
        raise RuntimeError(
            "No se ha podido validar robots.txt. "
            "Revisar la respuesta antes de continuar."
        )

    if esperar:
        tiempo_espera = random.uniform(ESPERA_MIN, ESPERA_MAX)
        time.sleep(tiempo_espera)

    respuesta = sesion.get(url, timeout=TIMEOUT)

    print("HTTP", respuesta.status_code, "-", url)

    if respuesta.status_code == 403:
        raise RuntimeError(
            "El servidor ha devuelto HTTP 403. "
            "Se detiene el scraping y no se intenta evitar el bloqueo."
        )

    if respuesta.status_code == 429:
        raise RuntimeError(
            "El servidor ha devuelto HTTP 429 (Too Many Requests). "
            "Se detiene el scraping."
        )

    respuesta.raise_for_status()

    return respuesta


## 3. Descargar el directorio e inspeccionar la estructura

El directorio completo de OSHWA contiene una tabla con los proyectos certificados.

Antes de automatizar la extracción conviene comprobar que:

- la página se descarga;
- se encuentra una tabla;
- la tabla contiene filas.

Esta inspección previa permite detectar rápidamente si la estructura HTML ha cambiado.


In [5]:
# Primera descarga del directorio.
#
# Bibliografía:
#              - https://www.geeksforgeeks.org/python/parsing-tables-and-xml-with-beautifulsoup/
#              - https://www.geeksforgeeks.org/python/implementing-web-scraping-python-beautiful-soup/
#
# GeeksForGeeks propone primero crear el objeto BeautifulSoup y localizar la tabla
# antes de recorrer sus filas. Aquí hago la misma comprobación sobre OSHWA.

respuesta_directorio = descargar_pagina(URL_DIRECTORIO, esperar=False)

soup_directorio = BeautifulSoup(
    respuesta_directorio.content,
    "html.parser"
)

print("Título HTML:")
print(soup_directorio.title.get_text(" ", strip=True) if soup_directorio.title else "No encontrado")

tabla = soup_directorio.find("table")

if tabla is None:
    print("No se ha encontrado ninguna tabla. Hay que revisar la estructura HTML.")
else:
    filas = tabla.find_all("tr")
    print("Tabla encontrada")
    print("Número de filas encontradas:", len(filas))

HTTP 200 - https://certification.oshwa.org/list.html
Título HTML:
OSHWA Certified Projects List
Tabla encontrada
Número de filas encontradas: 3460



## 4. Extraer los proyectos de la tabla

Cada fila contiene información básica del proyecto:

- nombre;
- UID;
- tipo principal;
- fecha de certificación;
- enlace a la ficha individual.

Utilizaremos `find_all("tr")` para recorrer las filas y `find_all("td")` para obtener las celdas.

Después buscaremos el primer enlace `<a href="...">` dentro de la primera celda.

**Bibliografía utilizada para esta celda:**

- GeeksForGeeks. Ejemplo prácticamente igual para recorrer una tabla HTML mediante `find('table')`, `find_all('tr')` y `find_all('td')`:  
  https://www.geeksforgeeks.org/python/parsing-tables-and-xml-with-beautifulsoup/

- Stack Overflow. Ejemplo que obtiene el primer `href` existente en cada fila de una tabla:  
  https://stackoverflow.com/questions/57066394/trying-to-grab-just-the-first-href-in-each-table-row

- Stack Overflow. Ejemplo de extracción de enlaces y conversión de enlaces relativos a absolutos utilizando `urllib.parse.urljoin`:  
  https://stackoverflow.com/questions/75904265/python-web-scraper-not-populating-txt-file-with-any-scraped-data


In [6]:
def extraer_proyectos_directorio(soup):
    """
    Recorre la tabla principal de OSHWA y devuelve una lista de diccionarios.
    """
    # Bibliografía:
    #              - https://www.geeksforgeeks.org/python/parsing-tables-and-xml-with-beautifulsoup/
    #              - https://stackoverflow.com/questions/57066394/trying-to-grab-just-the-first-href-in-each-table-row
    #              - https://stackoverflow.com/questions/75904265/python-web-scraper-not-populating-txt-file-with-any-scraped-data
    #
    # Adapto el patrón tabla -> filas -> celdas -> enlace al formato concreto de OSHWA.

    tabla = soup.find("table")

    if tabla is None:
        return []

    proyectos = []

    for fila in tabla.find_all("tr"):
        celdas = fila.find_all("td")

        # La cabecera no tiene td y algunas filas no válidas podrían no tener las 4 celdas esperadas.
        if len(celdas) < 4:
            continue

        enlace = celdas[0].find("a", href=True)

        if enlace is None:
            continue

        nombre = enlace.get_text(" ", strip=True)
        uid = celdas[1].get_text(" ", strip=True)
        tipo_principal = celdas[2].get_text(" ", strip=True)
        fecha_certificacion = celdas[3].get_text(" ", strip=True)

        # Los UID de OSHWA siguen un formato similar a US000001, ES000010, etc.
        # Lo utilizo como comprobación para evitar capturar enlaces de navegación.
        if not re.fullmatch(r"[A-Z]{2}\d{6}", uid):
            continue

        url_ficha = urljoin(URL_DIRECTORIO, enlace["href"])

        proyectos.append({
            "project_name_list": nombre,
            "uid": uid,
            "primary_project_type": tipo_principal,
            "certification_date_list": fecha_certificacion,
            "detail_url": url_ficha
        })

    return proyectos

In [7]:
# Aplicamos la función al directorio y observamos los primeros resultados.

proyectos_directorio = extraer_proyectos_directorio(soup_directorio)

print("Número de proyectos encontrados:", len(proyectos_directorio))

df_directorio = pd.DataFrame(proyectos_directorio)

display(df_directorio.head(10))

Número de proyectos encontrados: 3459


,project_name_list,uid,primary_project_type,certification_date_list,detail_url
0,OpenLog,US000001,Electronics,"October 01, 2016",https://certification.oshwa.org/us000001.html
1,blink(1) USB notification light,US000051,Electronics,"October 04, 2016",https://certification.oshwa.org/us000051.html
2,BeagleBone Black Wireless,US000011,Electronics,"October 06, 2016",https://certification.oshwa.org/us000011.html
3,ColorHug,UK000001,Electronics,"August 20, 2018",https://certification.oshwa.org/uk000001.html
4,HackEEG,US000027,Electronics,"October 08, 2016",https://certification.oshwa.org/us000027.html
5,Smoothieware,FR000001,Electronics,"October 08, 2016",https://certification.oshwa.org/fr000001.html
6,SAV MAKER I,ES000002,Electronics,"October 08, 2016",https://certification.oshwa.org/es000002.html
7,OpenEVSE,US000028,Electronics,"October 08, 2016",https://certification.oshwa.org/us000028.html
8,Open.Theremin,CH000001,Sound,"October 08, 2016",https://certification.oshwa.org/ch000001.html
9,OneRNG,NZ000001,Electronics,"October 08, 2016",https://certification.oshwa.org/nz000001.html



## 5. Filtrar proyectos de electrónica

Para esta primera versión quiero trabajar con proyectos relacionados directamente con electrónica.

El directorio ya incluye un campo `Project type`, por lo que utilizaremos el tipo principal para seleccionar únicamente `Electronics`.

El filtro solo afecta a qué fichas visitamos. El código de extracción del directorio continúa recogiendo todos los proyectos encontrados.


In [8]:
# Seleccionamos proyectos cuyo tipo principal sea Electronics.

if SOLO_ELECTRONICA:
    proyectos_seleccionados = [
        proyecto
        for proyecto in proyectos_directorio
        if proyecto["primary_project_type"].strip().lower() == "electronics"
    ]
else:
    proyectos_seleccionados = proyectos_directorio.copy()

print("Proyectos disponibles después del filtro:", len(proyectos_seleccionados))

# Para las pruebas trabajamos solo con los primeros registros.
if MAX_PROYECTOS is not None:
    proyectos_seleccionados = proyectos_seleccionados[:MAX_PROYECTOS]

print("Proyectos que se procesarán en esta ejecución:", len(proyectos_seleccionados))

pd.DataFrame(proyectos_seleccionados).head()

Proyectos disponibles después del filtro: 2465
Proyectos que se procesarán en esta ejecución: 20


,project_name_list,uid,primary_project_type,certification_date_list,detail_url
0,OpenLog,US000001,Electronics,"October 01, 2016",https://certification.oshwa.org/us000001.html
1,blink(1) USB notification light,US000051,Electronics,"October 04, 2016",https://certification.oshwa.org/us000051.html
2,BeagleBone Black Wireless,US000011,Electronics,"October 06, 2016",https://certification.oshwa.org/us000011.html
3,ColorHug,UK000001,Electronics,"August 20, 2018",https://certification.oshwa.org/uk000001.html
4,HackEEG,US000027,Electronics,"October 08, 2016",https://certification.oshwa.org/us000027.html



## 6. Entrar automáticamente en una ficha

Hasta ahora solo hemos trabajado con la tabla principal.

El requisito importante de la práctica es que exista **descubrimiento de enlaces y navegación autónoma**. Por tanto, utilizaremos el enlace obtenido en el directorio para entrar automáticamente en la página individual de cada proyecto.

Primero probaremos con una única ficha.

**Bibliografía utilizada para esta celda:**

- Stack Overflow. Pregunta muy similar: realizar una petición inicial, extraer los enlaces de una tabla y después hacer un bucle realizando una nueva petición para cada enlace:  
  https://stackoverflow.com/questions/70795634/how-can-beautifulsoup-scrape-the-pages-inside-this-list-of-hyperlinks

- Stack Overflow. Ejemplo con `requests.Session()`, `urljoin()` y posterior acceso a cada subpágina descubierta:  
  https://stackoverflow.com/questions/33135929/accessing-tabular-data-via-hyperlinks-with-beautifulsoup

- Stack Overflow. Ejemplo básico donde se obtiene una lista de enlaces y posteriormente se hace `requests.get()` de cada enlace:  
  https://stackoverflow.com/questions/29421840/how-to-scrape-a-page-with-beautifulsoup-and-python


In [9]:
# Probamos el proceso con el primer proyecto de la selección.
#
# Bibliografía:
#              - https://stackoverflow.com/questions/70795634/how-can-beautifulsoup-scrape-the-pages-inside-this-list-of-hyperlinks
#              - https://stackoverflow.com/questions/33135929/accessing-tabular-data-via-hyperlinks-with-beautifulsoup
#              - https://stackoverflow.com/questions/29421840/how-to-scrape-a-page-with-beautifulsoup-and-python

if proyectos_seleccionados:
    proyecto_prueba = proyectos_seleccionados[0]

    print("Proyecto:", proyecto_prueba["project_name_list"])
    print("Ficha:", proyecto_prueba["detail_url"])

    respuesta_ficha = descargar_pagina(proyecto_prueba["detail_url"])
    soup_ficha = BeautifulSoup(respuesta_ficha.content, "html.parser")

    print("\nTítulo de la ficha:")
    print(soup_ficha.find("h1").get_text(" ", strip=True) if soup_ficha.find("h1") else "No encontrado")

    print("\nEncabezados encontrados:")
    for encabezado in soup_ficha.find_all(["h2", "h3", "h4", "h5", "h6"]):
        print(encabezado.name, "-", encabezado.get_text(" ", strip=True))
else:
    print("No hay proyectos seleccionados.")

Proyecto: OpenLog
Ficha: https://certification.oshwa.org/us000001.html
HTTP 200 - https://certification.oshwa.org/us000001.html

Título de la ficha:
OpenLog

Encabezados encontrados:
h2 - Nathan Seidle mail
h3 - OSHWA UID
h3 - Certification Date
h3 - Country
h3 - Version
h5 - Licenses
h6 - Hardware
h6 - Software
h6 - Documentation
h5 - Certification mark
h5 - Project Type
h5 - Keywords



## 7. Funciones auxiliares para extraer valores situados después de encabezados

Las fichas de OSHWA están organizadas mediante encabezados como:

- `OSHWA UID`
- `Certification Date`
- `Country`
- `Version`
- `Previous Versions`
- `Licenses`
- `Project Type`
- `Keywords`

Una forma de extraer este tipo de información con BeautifulSoup consiste en localizar un encabezado y recorrer los elementos posteriores hasta llegar al siguiente encabezado.

**Bibliografía utilizada para esta celda:**

- Stack Overflow. Ejemplo que localiza encabezados `<h3>` y obtiene el elemento hermano situado a continuación mediante `find_next_sibling()`:  
  https://stackoverflow.com/questions/27193884/how-to-grab-alternating-child-tags-in-python-beautifulsoup

- GeeksForGeeks. Explicación y ejemplos de `nextSibling` / hermanos siguientes con BeautifulSoup:  
  https://www.geeksforgeeks.org/python/beautifulsoup-nextsibling/

- Stack Overflow. Ejemplo de extracción de parejas etiqueta/valor en estructuras HTML mediante `<dt>` y `<dd>`, utilizando `find_next()`:  
  https://stackoverflow.com/questions/62104956/how-can-i-extract-text-and-lists-with-beautifulsoup


In [10]:
# Funciones auxiliares para navegar por la estructura de cada ficha.
#
# Bibliografía:
#              - https://stackoverflow.com/questions/27193884/how-to-grab-alternating-child-tags-in-python-beautifulsoup
#              - https://www.geeksforgeeks.org/python/beautifulsoup-nextsibling/
#              - https://stackoverflow.com/questions/62104956/how-can-i-extract-text-and-lists-with-beautifulsoup

TAGS_TITULO = {"h1", "h2", "h3", "h4", "h5", "h6"}


def normalizar_texto(texto):
    """Elimina espacios repetidos y pasa el texto a minúsculas para comparar."""
    if texto is None:
        return ""
    return re.sub(r"\s+", " ", texto).strip().casefold()


def buscar_titulo(soup, texto):
    """Busca un encabezado h1-h6 cuyo texto coincida con el indicado."""
    objetivo = normalizar_texto(texto)

    return soup.find(
        lambda tag:
            tag.name in TAGS_TITULO
            and normalizar_texto(tag.get_text(" ", strip=True)) == objetivo
    )


def textos_hasta_siguiente_titulo(titulo):
    """
    Obtiene textos situados después de un encabezado hasta encontrar otro h1-h6.
    Se utilizan únicamente elementos sin hijos HTML para reducir textos duplicados.
    """
    if titulo is None:
        return []

    textos = []

    for elemento in titulo.find_all_next():
        if elemento is titulo:
            continue

        if elemento.name in TAGS_TITULO:
            break

        if elemento.name in {"script", "style"}:
            continue

        # Si el elemento contiene otras etiquetas, normalmente es un contenedor y
        # su texto aparecerá de nuevo en los elementos hijos.
        if elemento.find(True) is not None:
            continue

        texto = elemento.get_text(" ", strip=True)

        if texto and texto not in textos:
            textos.append(texto)

    return textos


def primer_valor_despues_de_titulo(soup, texto_titulo):
    """Devuelve el primer valor textual encontrado después de un encabezado."""
    titulo = buscar_titulo(soup, texto_titulo)
    valores = textos_hasta_siguiente_titulo(titulo)

    if valores:
        return valores[0]

    return None


## 8. Extraer los campos de una ficha de OSHWA

Ahora agruparemos la información de la ficha en un diccionario.

Se recogerán campos de distintos tipos:

- texto: nombre, creador, descripción;
- categóricos: país y licencias;
- fecha: fecha de certificación;
- listas: tipos de proyecto, keywords y versiones anteriores;
- URLs: web del proyecto, documentación y archivos gráficos del certificado.

Esto nos permitirá obtener un dataset bastante heterogéneo sin necesidad de limpiar todavía los datos.

Para los enlaces volvemos a utilizar `urljoin()`, ya que un `href` puede ser absoluto o relativo.

**Bibliografía adicional:**

- Stack Overflow. Uso de `urllib.parse.urljoin` para convertir un `href` relativo en URL completa:  
  https://stackoverflow.com/questions/69960877/python-beautifulsoup-how-can-i-get-full-link-from-href-attribute

- GeeksForGeeks. Extracción de enlaces mediante etiquetas `<a>` y atributo `href`:  
  https://www.geeksforgeeks.org/python/beautifulsoup-scraping-link-from-html/


In [11]:
TIPOS_PROYECTO_OSHWA = {
    "3d printing",
    "agriculture",
    "arts",
    "education",
    "electronics",
    "enclosure",
    "environmental",
    "home connection",
    "iot",
    "manufacturing",
    "other",
    "robotics",
    "science",
    "sound",
    "space",
    "tool",
    "wearables"
}


def buscar_enlace_por_texto(soup, texto):
    """Busca un enlace cuyo texto contenga la cadena indicada."""
    enlace = soup.find(
        "a",
        href=True,
        string=lambda s: s and texto.casefold() in s.casefold()
    )

    if enlace:
        return urljoin(BASE_URL, enlace["href"])

    # Algunos enlaces pueden contener etiquetas internas y string sería None.
    for enlace in soup.find_all("a", href=True):
        if texto.casefold() in enlace.get_text(" ", strip=True).casefold():
            return urljoin(BASE_URL, enlace["href"])

    return None


def extraer_descripcion(soup):
    """
    En las fichas de OSHWA la descripción aparece después de Country
    y antes de Version. Seleccionamos el texto más largo de ese bloque
    descartando el primer elemento, que normalmente es el nombre del país.
    """
    titulo_country = buscar_titulo(soup, "Country")
    valores = textos_hasta_siguiente_titulo(titulo_country)

    if len(valores) <= 1:
        return None

    candidatos = valores[1:]

    return max(candidatos, key=len) if candidatos else None


def extraer_tipos_proyecto(soup):
    """Recoge los tipos situados entre Project Type y Keywords."""
    titulo = buscar_titulo(soup, "Project Type")

    if titulo is None:
        return []

    tipos = []

    for elemento in titulo.find_all_next():
        if elemento is titulo:
            continue

        if elemento.name in TAGS_TITULO:
            texto_encabezado = normalizar_texto(elemento.get_text(" ", strip=True))

            if texto_encabezado == "keywords":
                break

            continue

        if elemento.find(True) is not None:
            continue

        texto = elemento.get_text(" ", strip=True)
        texto_normalizado = normalizar_texto(texto)

        if texto_normalizado in TIPOS_PROYECTO_OSHWA and texto not in tipos:
            tipos.append(texto)

    return tipos


def extraer_keywords(soup):
    """Recoge el texto situado después de Keywords y lo separa por comas."""
    titulo = buscar_titulo(soup, "Keywords")
    valores = textos_hasta_siguiente_titulo(titulo)

    if not valores:
        return []

    # Normalmente OSHWA muestra las keywords como una cadena separada por comas.
    texto_keywords = valores[0]

    return [
        keyword.strip()
        for keyword in texto_keywords.split(",")
        if keyword.strip()
    ]


def extraer_marcas_certificacion(soup, url_actual):
    """Busca enlaces SVG y PNG dentro del bloque Certification mark."""
    titulo = buscar_titulo(soup, "Certification mark")

    if titulo is None:
        return None, None

    svg_url = None
    png_url = None

    for elemento in titulo.find_all_next():
        if elemento is titulo:
            continue

        if elemento.name in TAGS_TITULO:
            if normalizar_texto(elemento.get_text(" ", strip=True)) == "project type":
                break

        if elemento.name == "a" and elemento.get("href"):
            enlace = urljoin(url_actual, elemento["href"])

            if re.search(r"\.svg(?:\?|$)", enlace, flags=re.I):
                svg_url = svg_url or enlace

            if re.search(r"\.png(?:\?|$)", enlace, flags=re.I):
                png_url = png_url or enlace

    return svg_url, png_url

In [12]:
def extraer_detalle_oshwa(soup, url_actual):
    """Extrae la información de una ficha individual de certificación OSHWA."""

    # Nombre del proyecto
    h1 = soup.find("h1")
    project_name = h1.get_text(" ", strip=True) if h1 else None

    # Autor / organización
    h2 = soup.find("h2")
    author = h2.get_text(" ", strip=True) if h2 else None

    # Correo, si se muestra mediante mailto
    email = None
    if h2:
        mail = h2.find("a", href=re.compile(r"^mailto:", re.I))
        if mail:
            email = unquote(mail["href"].split(":", 1)[1])
            author = author.replace(mail.get_text(" ", strip=True), "").strip(" ()-")

    # Valores organizados mediante encabezados
    certification_date = primer_valor_despues_de_titulo(soup, "Certification Date")
    country = primer_valor_despues_de_titulo(soup, "Country")
    version = primer_valor_despues_de_titulo(soup, "Version")

    # Versiones anteriores no aparecen en todos los proyectos.
    previous_text = primer_valor_despues_de_titulo(soup, "Previous Versions")
    previous_versions = []

    if previous_text:
        previous_versions = re.findall(r"[A-Z]{2}\d{6}", previous_text)

    # Licencias
    hardware_license = primer_valor_despues_de_titulo(soup, "Hardware")
    software_license = primer_valor_despues_de_titulo(soup, "Software")
    documentation_license = primer_valor_despues_de_titulo(soup, "Documentation")

    # Enlaces
    project_website = buscar_enlace_por_texto(soup, "Project Website")
    documentation_url = buscar_enlace_por_texto(soup, "View Documentation")

    # Descripción, tipos y keywords
    description = extraer_descripcion(soup)
    project_types = extraer_tipos_proyecto(soup)
    keywords = extraer_keywords(soup)

    # Recursos gráficos del certificado
    certification_svg_url, certification_png_url = extraer_marcas_certificacion(
        soup,
        url_actual
    )

    return {
        "project_name": project_name,
        "author": author,
        "author_email": email,
        "certification_date": certification_date,
        "country": country,
        "description": description,
        "version": version,
        "previous_versions": previous_versions,
        "hardware_license": hardware_license,
        "software_license": software_license,
        "documentation_license": documentation_license,
        "project_types": project_types,
        "keywords": keywords,
        "project_website": project_website,
        "documentation_url": documentation_url,
        "certification_svg_url": certification_svg_url,
        "certification_png_url": certification_png_url
    }


## 9. Probar la extracción con una única ficha

Antes de recorrer 20 o cientos de proyectos es mejor comprobar una sola ficha.

De esta forma podremos detectar rápidamente si alguno de los encabezados o selectores ha cambiado.


In [13]:
if proyectos_seleccionados:
    proyecto_prueba = proyectos_seleccionados[0]

    respuesta_ficha = descargar_pagina(proyecto_prueba["detail_url"])
    soup_ficha = BeautifulSoup(respuesta_ficha.content, "html.parser")

    detalle_prueba = extraer_detalle_oshwa(
        soup_ficha,
        proyecto_prueba["detail_url"]
    )

    print("Datos obtenidos de la ficha:\n")

    for campo, valor in detalle_prueba.items():
        print(campo, ":", valor)
else:
    print("No hay proyectos disponibles para realizar la prueba.")

HTTP 200 - https://certification.oshwa.org/us000001.html
Datos obtenidos de la ficha:

project_name : OpenLog
author : Nathan Seidle
author_email : support@sparkfun.com
certification_date : October 01, 2016
country : United States of America
description : The SparkFun OpenLog is an open source data logger that works over a simple serial connection and supports microSD cards up to 32GB.
version : 15
previous_versions : []
hardware_license : Other
software_license : Other
documentation_license : CC BY-SA
project_types : ['Electronics']
keywords : ['Arduino']
project_website : https://www.sparkfun.com/products/13712
documentation_url : https://learn.sparkfun.com/tutorials/openlog-hookup-guide
certification_svg_url : None
certification_png_url : None



## 10. Recorrer automáticamente todos los proyectos seleccionados

Una vez comprobado el funcionamiento de una ficha podemos recorrer la lista de enlaces.

Este es el punto donde realmente se produce la **navegación autónoma**:

1. el programa obtiene los enlaces desde el directorio;
2. no escribimos manualmente las URLs de los proyectos;
3. el programa entra en cada enlace descubierto;
4. extrae la información;
5. combina los campos del listado con los de la ficha.

**Bibliografía utilizada para esta celda:**

- Stack Overflow. Ejemplo casi idéntico: obtener una lista de enlaces y recorrerla realizando una petición y un nuevo `BeautifulSoup` para cada URL:  
  https://stackoverflow.com/questions/70795634/how-can-beautifulsoup-scrape-the-pages-inside-this-list-of-hyperlinks

- Stack Overflow. Ejemplo que obtiene enlaces relativos de una página y realiza una petición posterior sobre cada uno:  
  https://stackoverflow.com/questions/29421840/how-to-scrape-a-page-with-beautifulsoup-and-python

- GeeksForGeeks. Ejemplo que recorre una lista de URLs, realiza una petición por URL y acumula los resultados antes de crear el DataFrame:  
  https://www.geeksforgeeks.org/python/web-scraping-for-stock-prices-in-python/


In [14]:
def ejecutar_scraping(proyectos):
    """
    Recorre las fichas descubiertas automáticamente en el directorio
    y devuelve una lista de registros.
    """
    # Bibliografía:
    #              - https://stackoverflow.com/questions/70795634/how-can-beautifulsoup-scrape-the-pages-inside-this-list-of-hyperlinks
    #              - https://stackoverflow.com/questions/29421840/how-to-scrape-a-page-with-beautifulsoup-and-python
    #              - https://www.geeksforgeeks.org/python/web-scraping-for-stock-prices-in-python/

    resultados = []

    total = len(proyectos)

    for numero, proyecto in enumerate(proyectos, start=1):
        print("\n" + "=" * 80)
        print(f"Proyecto {numero}/{total}")
        print(proyecto["uid"], "-", proyecto["project_name_list"])

        try:
            respuesta = descargar_pagina(proyecto["detail_url"])
            soup = BeautifulSoup(respuesta.content, "html.parser")

            detalle = extraer_detalle_oshwa(
                soup,
                proyecto["detail_url"]
            )

            # Unimos los campos obtenidos del listado con los obtenidos en la ficha.
            registro = {
                **proyecto,
                **detalle
            }

            resultados.append(registro)

        except (requests.RequestException, PermissionError, RuntimeError) as error:
            print("No se ha podido procesar el proyecto:", error)

    return resultados

In [15]:
# Ejecutamos el scraping de la selección preparada anteriormente.

datos_oshwa = ejecutar_scraping(proyectos_seleccionados)

print("\nNúmero de fichas procesadas correctamente:", len(datos_oshwa))


Proyecto 1/20
US000001 - OpenLog
HTTP 200 - https://certification.oshwa.org/us000001.html

Proyecto 2/20
US000051 - blink(1) USB notification light
HTTP 200 - https://certification.oshwa.org/us000051.html

Proyecto 3/20
US000011 - BeagleBone Black Wireless
HTTP 200 - https://certification.oshwa.org/us000011.html

Proyecto 4/20
UK000001 - ColorHug
HTTP 200 - https://certification.oshwa.org/uk000001.html

Proyecto 5/20
US000027 - HackEEG
HTTP 200 - https://certification.oshwa.org/us000027.html

Proyecto 6/20
FR000001 - Smoothieware
HTTP 200 - https://certification.oshwa.org/fr000001.html

Proyecto 7/20
ES000002 - SAV MAKER I
HTTP 200 - https://certification.oshwa.org/es000002.html

Proyecto 8/20
US000028 - OpenEVSE
HTTP 200 - https://certification.oshwa.org/us000028.html

Proyecto 9/20
NZ000001 - OneRNG
HTTP 200 - https://certification.oshwa.org/nz000001.html

Proyecto 10/20
MX000001 - Byteman
HTTP 200 - https://certification.oshwa.org/mx000001.html

Proyecto 11/20
AT000001 - AXIOM
HTTP


## 11. Crear el DataFrame

Durante el scraping hemos ido guardando cada proyecto como un diccionario.

Ahora utilizamos directamente esa **lista de diccionarios** para crear un `DataFrame`.

**Bibliografía utilizada para esta celda:**

- GeeksForGeeks. Ejemplo específico de creación de un DataFrame a partir de una lista de diccionarios:  
  https://www.geeksforgeeks.org/python/create-a-pandas-dataframe-from-list-of-dicts/

- GeeksForGeeks. Ejemplo de scraping donde cada elemento se guarda en un diccionario y se acumula en una lista:  
  https://www.geeksforgeeks.org/python/implementing-web-scraping-python-beautiful-soup/


In [16]:
# Convertimos la lista de diccionarios a DataFrame.
#
# Bibliografía:
#              - https://www.geeksforgeeks.org/python/create-a-pandas-dataframe-from-list-of-dicts/
#              - https://www.geeksforgeeks.org/python/implementing-web-scraping-python-beautiful-soup/

df_oshwa = pd.DataFrame(datos_oshwa)

print("Número de registros:", len(df_oshwa))
print("Número de columnas:", len(df_oshwa.columns))

display(df_oshwa.head())

Número de registros: 20
Número de columnas: 22


,project_name_list,uid,primary_project_type,certification_date_list,detail_url,project_name,author,author_email,certification_date,country,...,previous_versions,hardware_license,software_license,documentation_license,project_types,keywords,project_website,documentation_url,certification_svg_url,certification_png_url
0,OpenLog,US000001,Electronics,"October 01, 2016",https://certification.oshwa.org/us000001.html,OpenLog,Nathan Seidle,support@sparkfun.com,"October 01, 2016",United States of America,...,[],Other,Other,CC BY-SA,[Electronics],[Arduino],https://www.sparkfun.com/products/13712,https://learn.sparkfun.com/tutorials/openlog-h...,None,None
1,blink(1) USB notification light,US000051,Electronics,"October 04, 2016",https://certification.oshwa.org/us000051.html,blink(1) USB notification light,ThingM,info@thingm.com,"October 04, 2016",United States of America,...,[],Other,Other,CC BY-SA,[Electronics],"[USB, light]",https://blink1.thingm.com/,https://github.com/todbot/blink1/blob/master/L...,None,None
2,BeagleBone Black Wireless,US000011,Electronics,"October 06, 2016",https://certification.oshwa.org/us000011.html,BeagleBone Black Wireless,BeagleBoard.org Foundation,oshwa@beagleboard.org,"October 06, 2016",United States of America,...,[],Other,Other,CC BY,"[Electronics, Education]",[Computer],https://beagleboard.org/black-wireless,https://github.com/beagleboard/BeagleBone-blac...,None,None
3,ColorHug,UK000001,Electronics,"August 20, 2018",https://certification.oshwa.org/uk000001.html,ColorHug,Hughski Limited,NaN,"August 20, 2018",United Kingdom,...,[],CERN,GPL,CC BY-SA,"[Electronics, Arts, Tool]","[color, calibration, display]",http://www.hughski.com/,https://hughski.com/faq.html#openhardware,None,None
4,HackEEG,US000027,Electronics,"October 08, 2016",https://certification.oshwa.org/us000027.html,HackEEG,"StarCat, LLC",info@starcat.io,"October 08, 2016",United States of America,...,[],Other,Other,CC BY-SA,"[Electronics, Robotics]","[EEG, Arduino, shield]",https://starcat.io/products/hackeeg-shield/,https://github.com/adamfeuer/hackeeg-shield,None,None



## 12. Construir un texto técnico para cada proyecto

Hasta este punto los datos proceden directamente de la web.

Para orientar el dataset hacia el análisis de tecnologías crearemos ahora un campo auxiliar denominado `technical_text`, combinando:

- nombre del proyecto;
- descripción;
- keywords publicadas en la ficha.

No se elimina ni sustituye ninguno de los campos originales. El objetivo es disponer de un único texto sobre el que buscar posteriormente nombres de tecnologías.

Esta operación es una transformación posterior al scraping y se mantiene separada del proceso de captura.


In [17]:
# Unimos nombre, descripción y keywords en un único texto.
#
# Este campo se utiliza únicamente para detectar tecnologías mencionadas.
# Los campos originales se conservan sin modificar.

def construir_texto_tecnico(fila):
    partes = []

    for columna in ["project_name", "description"]:
        valor = fila.get(columna)

        if pd.notna(valor) and str(valor).strip():
            partes.append(str(valor))

    keywords = fila.get("keywords")

    if isinstance(keywords, list):
        partes.extend(str(k) for k in keywords)
    elif pd.notna(keywords) and str(keywords).strip():
        partes.append(str(keywords))

    return " ".join(partes)


if not df_oshwa.empty:
    df_oshwa["technical_text"] = df_oshwa.apply(
        construir_texto_tecnico,
        axis=1
    )

    display(
        df_oshwa[
            ["uid", "project_name", "technical_text"]
        ].head()
    )

,uid,project_name,technical_text
0,US000001,OpenLog,OpenLog The SparkFun OpenLog is an open source...
1,US000051,blink(1) USB notification light,blink(1) USB notification light blink(1) is a ...
2,US000011,BeagleBone Black Wireless,BeagleBone Black Wireless Open hardware comput...
3,UK000001,ColorHug,ColorHug A display colorimeter. color calibrat...
4,US000027,HackEEG,HackEEG TI ADS1299 EEG SoC shield for Arduino ...



## 13. Detectar tecnologías mencionadas

Se define un pequeño diccionario de tecnologías de interés con diferentes formas de escritura.

Por ejemplo:

- `ESP32` puede aparecer como `ESP32`, `ESP32-S3`, `ESP32-C3`, etc.
- `RS-485` puede aparecer como `RS485` o `RS-485`.
- `Bluetooth Low Energy` puede aparecer como `BLE`.
- `LoRa` puede aparecer también en proyectos que mencionan `LoRaWAN`.

El objetivo **no es afirmar que todos estos elementos pertenecen a la BOM**, sino detectar tecnologías que el propio proyecto menciona en su nombre, descripción o keywords.

Para realizar la búsqueda utilizaremos expresiones regulares con `re.search()` y `re.IGNORECASE`.

**Bibliografía utilizada para esta celda:**

- Stack Overflow. Ejemplo que construye una expresión regular a partir de una lista de tecnologías y recupera coincidencias con `re.findall()` y `re.IGNORECASE`:  
  https://stackoverflow.com/questions/51487012/how-to-print-matched-keywords-from-two-lists

- Stack Overflow. Ejemplo de búsqueda de una lista de keywords utilizando `re.search()` / `re.findall()` de forma insensible a mayúsculas y minúsculas:  
  https://stackoverflow.com/questions/66523520/python-how-to-use-list-of-keywords-to-search-for-a-string-in-a-text

- Stack Overflow. Ejemplo de búsqueda de varias palabras mediante el operador OR (`|`) y `re.I`:  
  https://stackoverflow.com/questions/15650244/using-regex-in-python-2-7-3-to-search-text-and-output-matches

- Documentación oficial de Python sobre expresiones regulares:  
  https://docs.python.org/3/library/re.html


In [18]:
# Diccionario de tecnologías y patrones habituales de escritura.
#
# Bibliografía:
#              - https://stackoverflow.com/questions/51487012/how-to-print-matched-keywords-from-two-lists
#              - https://stackoverflow.com/questions/66523520/python-how-to-use-list-of-keywords-to-search-for-a-string-in-a-text
#              - https://stackoverflow.com/questions/15650244/using-regex-in-python-2-7-3-to-search-text-and-output-matches
#              - https://docs.python.org/3/library/re.html
#
# Se adapta el patrón "lista de palabras -> expresión regular -> búsqueda"
# al vocabulario técnico de electrónica.

PATRONES_TECNOLOGIA = {
    "ESP32": [
        r"\bESP32(?:[- ][A-Z0-9]+)?\b"
    ],
    "ESP8266": [
        r"\bESP8266\b"
    ],
    "STM32": [
        r"\bSTM32[A-Z0-9-]*\b"
    ],
    "RP2040": [
        r"\bRP2040\b",
        r"\bRaspberry\s+Pi\s+Pico\b"
    ],
    "Raspberry Pi": [
        r"\bRaspberry\s+Pi\b"
    ],
    "Arduino": [
        r"\bArduino\b"
    ],
    "ATmega": [
        r"\bATmega[A-Z0-9-]*\b"
    ],
    "ATtiny": [
        r"\bATtiny[A-Z0-9-]*\b"
    ],
    "nRF52": [
        r"\bnRF52[A-Z0-9-]*\b",
        r"\bnRF528[0-9A-Z-]*\b"
    ],
    "RISC-V": [
        r"\bRISC[- ]?V\b"
    ],
    "FPGA": [
        r"\bFPGA\b"
    ],
    "LoRa": [
        r"\bLoRa(?:WAN)?\b"
    ],
    "CAN": [
        r"\bCAN\s*bus\b",
        r"\bCANbus\b"
    ],
    "RS-485": [
        r"\bRS[- ]?485\b"
    ],
    "RS-232": [
        r"\bRS[- ]?232\b"
    ],
    "Ethernet": [
        r"\bEthernet\b"
    ],
    "Wi-Fi": [
        r"\bWi[- ]?Fi\b"
    ],
    "Bluetooth/BLE": [
        r"\bBluetooth\b",
        r"\bBLE\b",
        r"\bBluetooth\s+Low\s+Energy\b"
    ],
    "Zigbee": [
        r"\bZigbee\b"
    ],
    "USB": [
        r"\bUSB(?:[- ][A-Z0-9.]+)?\b"
    ],
    "I2C": [
        r"\bI2C\b",
        r"\bI²C\b"
    ],
    "SPI": [
        r"\bSPI\b"
    ],
    "UART": [
        r"\bUART\b"
    ],
    "GNSS/GPS": [
        r"\bGNSS\b",
        r"\bGPS\b"
    ],
    "IMU": [
        r"\bIMU\b"
    ]
}


def detectar_tecnologias(texto):
    """Devuelve las tecnologías del diccionario encontradas en el texto."""

    if texto is None or pd.isna(texto):
        return []

    texto = str(texto)
    encontradas = []

    for tecnologia, patrones in PATRONES_TECNOLOGIA.items():

        if any(
            re.search(patron, texto, flags=re.IGNORECASE)
            for patron in patrones
        ):
            encontradas.append(tecnologia)

    return encontradas


if not df_oshwa.empty:
    df_oshwa["technology_mentions"] = df_oshwa["technical_text"].apply(
        detectar_tecnologias
    )

    df_oshwa["technology_count"] = df_oshwa["technology_mentions"].apply(len)

    display(
        df_oshwa[
            [
                "uid",
                "project_name",
                "technology_mentions",
                "technology_count"
            ]
        ].head(15)
    )

,uid,project_name,technology_mentions,technology_count
0,US000001,OpenLog,[Arduino],1
1,US000051,blink(1) USB notification light,[USB],1
2,US000011,BeagleBone Black Wireless,[],0
3,UK000001,ColorHug,[],0
4,US000027,HackEEG,[Arduino],1
5,FR000001,Smoothieware,[],0
6,ES000002,SAV MAKER I,[Arduino],1
7,US000028,OpenEVSE,[],0
8,NZ000001,OneRNG,[],0
9,MX000001,Byteman,[],0



## 14. Buscar candidatos a referencias de componentes

Además de las tecnologías conocidas del diccionario anterior, puede ser interesante conservar posibles **referencias técnicas** que aparezcan en el texto.

Muchas referencias electrónicas combinan letras y números, por ejemplo:

- `BME280`
- `STM32G030`
- `RP2040`
- `ESP32-S3`
- `DHT11`
- `MCP2515`
- `MAX31865`

No todos los tokens con letras y números serán componentes reales. Por este motivo esta columna se denominará `candidate_technical_tokens`.

Precisamente esta información puede resultar interesante para una práctica posterior de limpieza, donde se podrían eliminar versiones, medidas u otros falsos positivos.

**Bibliografía utilizada para esta celda:**

- Stack Overflow. Uso de `re.findall()` para recuperar todas las coincidencias de un patrón dentro de un texto:  
  https://stackoverflow.com/questions/51487012/how-to-print-matched-keywords-from-two-lists

- GeeksForGeeks. Ejemplo de `re.finditer()` / expresiones regulares para localizar palabras dentro de una cadena:  
  https://www.geeksforgeeks.org/python/python-program-to-find-start-and-end-indices-of-all-words-in-a-string/

- Documentación oficial de Python `re.findall`:  
  https://docs.python.org/3/library/re.html#re.findall


In [19]:
# Detectamos tokens técnicos formados por letras y números.
#
# No se consideran todavía componentes confirmados.
# Es una extracción exploratoria que se podrá limpiar en una fase posterior.
#
# Bibliografía:
#              - https://stackoverflow.com/questions/51487012/how-to-print-matched-keywords-from-two-lists
#              - https://www.geeksforgeeks.org/python/python-program-to-find-start-and-end-indices-of-all-words-in-a-string/
#              - https://docs.python.org/3/library/re.html#re.findall

PATRON_TOKEN_TECNICO = re.compile(
    r"\b"
    r"(?=[A-Za-z0-9._+-]{4,}\b)"
    r"(?=[A-Za-z0-9._+-]*[A-Za-z])"
    r"(?=[A-Za-z0-9._+-]*\d)"
    r"[A-Za-z0-9][A-Za-z0-9._+-]*"
    r"\b"
)


def extraer_tokens_tecnicos(texto):
    if texto is None or pd.isna(texto):
        return []

    tokens = PATRON_TOKEN_TECNICO.findall(str(texto))

    # Quitamos duplicados conservando el orden de aparición.
    return list(dict.fromkeys(tokens))


if not df_oshwa.empty:
    df_oshwa["candidate_technical_tokens"] = df_oshwa["technical_text"].apply(
        extraer_tokens_tecnicos
    )

    df_oshwa["candidate_token_count"] = df_oshwa[
        "candidate_technical_tokens"
    ].apply(len)

    display(
        df_oshwa[
            [
                "uid",
                "project_name",
                "candidate_technical_tokens"
            ]
        ].head(15)
    )

,uid,project_name,candidate_technical_tokens
0,US000001,OpenLog,[32GB]
1,US000051,blink(1) USB notification light,[]
2,US000011,BeagleBone Black Wireless,[]
3,UK000001,ColorHug,[]
4,US000027,HackEEG,[ADS1299]
5,FR000001,Smoothieware,[]
6,ES000002,SAV MAKER I,[]
7,US000028,OpenEVSE,[]
8,NZ000001,OneRNG,[]
9,MX000001,Byteman,[]



## 15. Crear indicadores booleanos de tecnologías

Para facilitar análisis posteriores añadiremos una columna booleana por cada tecnología del diccionario.

Ejemplo:

| project | ESP32 | STM32 | LoRa | CAN |
|---|---:|---:|---:|---:|
| Proyecto A | True | False | True | False |
| Proyecto B | False | True | False | True |

Esta representación permitiría posteriormente:

- calcular frecuencias;
- estudiar combinaciones de tecnologías;
- construir vectores de características;
- aplicar clustering;
- desarrollar sistemas de recomendación basados en similitud.

En esta práctica únicamente crearemos las variables. El análisis o entrenamiento de modelos quedaría fuera del alcance actual.


In [20]:
# Creamos una variable booleana para cada tecnología detectada.
#
# El procedimiento es equivalente a comprobar si un elemento pertenece
# a la lista de coincidencias obtenida para cada registro.

if not df_oshwa.empty:

    for tecnologia in PATRONES_TECNOLOGIA:

        nombre_columna = (
            "uses_"
            + re.sub(
                r"[^a-z0-9]+",
                "_",
                tecnologia.casefold()
            ).strip("_")
        )

        df_oshwa[nombre_columna] = df_oshwa[
            "technology_mentions"
        ].apply(
            lambda lista: tecnologia in lista
        )

    columnas_uses = [
        columna
        for columna in df_oshwa.columns
        if columna.startswith("uses_")
    ]

    display(
        df_oshwa[
            ["uid", "project_name"] + columnas_uses[:10]
        ].head()
    )

,uid,project_name,uses_esp32,uses_esp8266,uses_stm32,uses_rp2040,uses_raspberry_pi,uses_arduino,uses_atmega,uses_attiny,uses_nrf52,uses_risc_v
0,US000001,OpenLog,False,False,False,False,False,True,False,False,False,False
1,US000051,blink(1) USB notification light,False,False,False,False,False,False,False,False,False,False
2,US000011,BeagleBone Black Wireless,False,False,False,False,False,False,False,False,False,False
3,UK000001,ColorHug,False,False,False,False,False,False,False,False,False,False
4,US000027,HackEEG,False,False,False,False,False,True,False,False,False,False



## 16. Comprobar qué tecnologías aparecen en la muestra

Esta tabla es únicamente una comprobación del resultado de la extracción.

No pretende ser todavía un análisis estadístico definitivo, ya que la muestra inicial está limitada por `MAX_PROYECTOS`.


In [21]:
if not df_oshwa.empty:

    frecuencia_tecnologias = (
        df_oshwa["technology_mentions"]
        .explode()
        .dropna()
        .value_counts()
        .rename_axis("technology")
        .reset_index(name="projects")
    )

    display(frecuencia_tecnologias)

,technology,projects
0,Arduino,3
1,USB,1
2,I2C,1
3,RS-232,1



## 17. Posible aplicación futura: recomendación de tecnologías y componentes

El dataset obtenido no es todavía suficiente para recomendar automáticamente una BOM completa, porque detectar que un proyecto menciona `ESP32`, `LoRa` o `CAN` no equivale a conocer todos los componentes utilizados.

Sin embargo, sí permite construir una primera relación:

**descripción del proyecto → tecnologías utilizadas**

Por ejemplo:

`"sensor ambiental inalámbrico alimentado por batería"`  
→ `ESP32`, `LoRa`, `I2C`

Una ampliación posterior podría seguir los enlaces de documentación que OSHWA obliga a publicar y extraer las **BOM** reales. Después esas referencias podrían enriquecerse con información técnica de fabricantes o distribuidores.

El dataset ampliado podría tener una estructura conceptual como:

**Descripción del proyecto → tecnologías → componentes/BOM → características técnicas**

Con suficientes ejemplos sería posible estudiar sistemas de:

- recuperación de proyectos similares;
- recomendación de tecnologías;
- recomendación de componentes;
- sustitución de componentes;
- clustering de arquitecturas hardware.

Por ahora esta posibilidad se utiliza únicamente como **motivación del dataset** y no forma parte del trabajo de machine learning de esta práctica.



## 18. Comprobaciones básicas del scraping

Todavía no realizamos la limpieza del dataset porque este proceso corresponderá a la Práctica 2.

Sí podemos hacer comprobaciones básicas para verificar que la extracción ha funcionado:

- número de registros;
- número de UID diferentes;
- columnas obtenidas;
- cantidad de valores nulos;
- posibles UID duplicados.

Estas comprobaciones no modifican los datos obtenidos.


In [22]:
print("Registros:", len(df_oshwa))

if not df_oshwa.empty:
    print("UID diferentes:", df_oshwa["uid"].nunique())
    print("UID duplicados:", df_oshwa["uid"].duplicated().sum())

    print("\nColumnas obtenidas:")
    print(df_oshwa.columns.tolist())

    print("\nValores nulos por columna:")
    display(df_oshwa.isna().sum().to_frame("nulos"))
else:
    print("El DataFrame está vacío.")

Registros: 20
UID diferentes: 20
UID duplicados: 0

Columnas obtenidas:
['project_name_list', 'uid', 'primary_project_type', 'certification_date_list', 'detail_url', 'project_name', 'author', 'author_email', 'certification_date', 'country', 'description', 'version', 'previous_versions', 'hardware_license', 'software_license', 'documentation_license', 'project_types', 'keywords', 'project_website', 'documentation_url', 'certification_svg_url', 'certification_png_url', 'technical_text', 'technology_mentions', 'technology_count', 'candidate_technical_tokens', 'candidate_token_count', 'uses_esp32', 'uses_esp8266', 'uses_stm32', 'uses_rp2040', 'uses_raspberry_pi', 'uses_arduino', 'uses_atmega', 'uses_attiny', 'uses_nrf52', 'uses_risc_v', 'uses_fpga', 'uses_lora', 'uses_can', 'uses_rs_485', 'uses_rs_232', 'uses_ethernet', 'uses_wi_fi', 'uses_bluetooth_ble', 'uses_zigbee', 'uses_usb', 'uses_i2c', 'uses_spi', 'uses_uart', 'uses_gnss_gps', 'uses_imu']

Valores nulos por columna:


,nulos
project_name_list,0
uid,0
primary_project_type,0
certification_date_list,0
detail_url,0
project_name,0
author,0
author_email,2
certification_date,0
country,0



## 19. Preparar las listas para guardar el CSV

Algunos campos contienen listas:

- `previous_versions`
- `project_types`
- `keywords`

CSV es un formato tabular, por lo que antes de exportar convertiremos estas listas en cadenas separadas mediante `|`.

Ejemplo:

`Electronics|IOT|Robotics`

Todavía no estamos limpiando el contenido; únicamente estamos cambiando su representación para poder almacenarlo en una celda del CSV.


In [23]:
df_export = df_oshwa.copy()

columnas_lista = [
    "previous_versions",
    "project_types",
    "keywords",
    "technology_mentions",
    "candidate_technical_tokens"
]

for columna in columnas_lista:
    if columna in df_export.columns:
        df_export[columna] = df_export[columna].apply(
            lambda valor:
                "|".join(valor)
                if isinstance(valor, list)
                else valor
        )

display(df_export.head())

,project_name_list,uid,primary_project_type,certification_date_list,detail_url,project_name,author,author_email,certification_date,country,...,uses_ethernet,uses_wi_fi,uses_bluetooth_ble,uses_zigbee,uses_usb,uses_i2c,uses_spi,uses_uart,uses_gnss_gps,uses_imu
0,OpenLog,US000001,Electronics,"October 01, 2016",https://certification.oshwa.org/us000001.html,OpenLog,Nathan Seidle,support@sparkfun.com,"October 01, 2016",United States of America,...,False,False,False,False,False,False,False,False,False,False
1,blink(1) USB notification light,US000051,Electronics,"October 04, 2016",https://certification.oshwa.org/us000051.html,blink(1) USB notification light,ThingM,info@thingm.com,"October 04, 2016",United States of America,...,False,False,False,False,True,False,False,False,False,False
2,BeagleBone Black Wireless,US000011,Electronics,"October 06, 2016",https://certification.oshwa.org/us000011.html,BeagleBone Black Wireless,BeagleBoard.org Foundation,oshwa@beagleboard.org,"October 06, 2016",United States of America,...,False,False,False,False,False,False,False,False,False,False
3,ColorHug,UK000001,Electronics,"August 20, 2018",https://certification.oshwa.org/uk000001.html,ColorHug,Hughski Limited,NaN,"August 20, 2018",United Kingdom,...,False,False,False,False,False,False,False,False,False,False
4,HackEEG,US000027,Electronics,"October 08, 2016",https://certification.oshwa.org/us000027.html,HackEEG,"StarCat, LLC",info@starcat.io,"October 08, 2016",United States of America,...,False,False,False,False,False,False,False,False,False,False



## 20. Guardar el dataset en CSV

Finalmente exportamos el primer dataset utilizando `DataFrame.to_csv()`.

**Bibliografía utilizada para esta celda:**

- GeeksForGeeks. Ejemplo directo de `df.to_csv(..., index=False)`:  
  https://www.geeksforgeeks.org/pandas/saving-a-pandas-dataframe-as-a-csv/

- GeeksForGeeks. Ejemplo que parte de una lista de diccionarios, crea el DataFrame y lo guarda en CSV:  
  https://www.geeksforgeeks.org/python/working-csv-files-python/


In [24]:
# Guardamos el dataset.
#
# Bibliografía:
#              - https://www.geeksforgeeks.org/pandas/saving-a-pandas-dataframe-as-a-csv/
#              - https://www.geeksforgeeks.org/python/working-csv-files-python/

nombre_csv = "oshwa_electronics_technologies_dataset_v1.csv"

df_export.to_csv(
    nombre_csv,
    index=False,
    encoding="utf-8"
)

print("Dataset guardado:", nombre_csv)
print("Número de registros:", len(df_export))

Dataset guardado: oshwa_electronics_technologies_dataset_v1.csv
Número de registros: 20



## Resumen de la primera versión del scraper

En esta primera versión se ha desarrollado el proceso completo de navegación:

- Se intenta comprobar `robots.txt`.
- Se utiliza un `User-Agent` identificable.
- Se utiliza `requests.Session()`.
- Se añaden pausas entre peticiones.
- Se detiene el proceso ante respuestas `403` o `429`.
- Se descarga el directorio real de proyectos certificados de OSHWA.
- Se localiza la tabla HTML.
- Se recorren automáticamente sus filas.
- Se descubren los enlaces a las fichas individuales.
- Se filtran inicialmente los proyectos cuyo tipo principal es `Electronics`.
- Se entra automáticamente en cada ficha descubierta.
- Se obtienen campos textuales, categóricos, fechas, listas y enlaces.
- Se genera un DataFrame.
- Se exporta el resultado a CSV.

Para la primera ejecución se utiliza un límite pequeño de proyectos. Una vez comprobado el funcionamiento, este límite se puede aumentar de manera controlada.


Además, esta versión añade una capa de enriquecimiento orientada al objetivo analítico:

- combina nombre, descripción y keywords en un texto técnico;
- detecta menciones de tecnologías mediante expresiones regulares;
- busca tokens que podrían corresponder a referencias electrónicas;
- crea indicadores booleanos por tecnología;
- conserva tanto los datos originales como las variables derivadas.

Es importante diferenciar entre **datos extraídos de OSHWA** y **variables derivadas posteriormente**. Las columnas `technology_mentions`, `candidate_technical_tokens` y `uses_*` son construidas por nuestro código y no proceden directamente de la web.



## Bibliografía utilizada en el desarrollo del web scraping

### Material de la asignatura

- Subirats Maté, L.; Calvo González, M. *Web scraping*. UOC.
- Lawson, R. *Web Scraping with Python*. Packt Publishing.

### Sitio web objeto del scraping

- OSHWA Certified Projects List:  
  https://certification.oshwa.org/list.html

- OSHWA Certified Projects Directory:  
  https://certification.oshwa.org/directory.html

### Ejemplos de Stack Overflow adaptados

- Descargar `robots.txt` manualmente y analizarlo con `RobotFileParser.parse()`:  
  https://stackoverflow.com/questions/37934316/change-user-agent-used-with-robotparser-in-python

- Crear una sesión, extraer enlaces y acceder posteriormente a las subpáginas:  
  https://stackoverflow.com/questions/33135929/accessing-tabular-data-via-hyperlinks-with-beautifulsoup

- Extraer el primer `href` de cada fila de una tabla:  
  https://stackoverflow.com/questions/57066394/trying-to-grab-just-the-first-href-in-each-table-row

- Obtener enlaces relativos y convertirlos a absolutos mediante `urljoin`:  
  https://stackoverflow.com/questions/75904265/python-web-scraper-not-populating-txt-file-with-any-scraped-data

- Recorrer una lista de enlaces y scrapear cada página individual:  
  https://stackoverflow.com/questions/70795634/how-can-beautifulsoup-scrape-the-pages-inside-this-list-of-hyperlinks

- Acceder mediante `requests.get()` a cada enlace obtenido previamente:  
  https://stackoverflow.com/questions/29421840/how-to-scrape-a-page-with-beautifulsoup-and-python

- Buscar un encabezado y acceder al elemento situado después:  
  https://stackoverflow.com/questions/27193884/how-to-grab-alternating-child-tags-in-python-beautifulsoup

- Extraer parejas de etiquetas y valores mediante BeautifulSoup:  
  https://stackoverflow.com/questions/62104956/how-can-i-extract-text-and-lists-with-beautifulsoup

- Convertir un enlace relativo a una URL completa mediante `urllib.parse.urljoin`:  
  https://stackoverflow.com/questions/69960877/python-beautifulsoup-how-can-i-get-full-link-from-href-attribute

### Ejemplos de GeeksForGeeks adaptados

- Recorrer y extraer los datos de una tabla HTML con BeautifulSoup:  
  https://www.geeksforgeeks.org/python/parsing-tables-and-xml-with-beautifulsoup/

- Extraer enlaces de etiquetas `<a>` mediante `href`:  
  https://www.geeksforgeeks.org/python/beautifulsoup-scraping-link-from-html/

- Scraping de varias páginas introduciendo pausas entre peticiones:  
  https://www.geeksforgeeks.org/python/how-to-scrape-multiple-pages-of-a-website-using-python/

- Scraping de distintas URLs con cabeceras y `time.sleep()`:  
  https://www.geeksforgeeks.org/python/web-scraping-for-stock-prices-in-python/

- Uso de hermanos siguientes en BeautifulSoup:  
  https://www.geeksforgeeks.org/python/beautifulsoup-nextsibling/

- Creación de DataFrame desde una lista de diccionarios:  
  https://www.geeksforgeeks.org/python/create-a-pandas-dataframe-from-list-of-dicts/

- Guardar un DataFrame en CSV:  
  https://www.geeksforgeeks.org/pandas/saving-a-pandas-dataframe-as-a-csv/

- Escritura de CSV con Pandas:  
  https://www.geeksforgeeks.org/python/working-csv-files-python/

### Documentación oficial

- Requests:  
  https://requests.readthedocs.io/en/latest/

- Requests Sessions:  
  https://requests.readthedocs.io/en/latest/user/advanced/

- BeautifulSoup:  
  https://www.crummy.com/software/BeautifulSoup/bs4/doc/

- Python `urllib.robotparser`:  
  https://docs.python.org/3/library/urllib.robotparser.html

- Python `urllib.parse.urljoin`:  
  https://docs.python.org/3/library/urllib.parse.html#urllib.parse.urljoin


### Fuentes sobre OSHWA y la motivación del dataset

- OSHWA - About:  
  https://oshwa.org/about/

- Open Source Hardware Definition:  
  https://oshwa.org/resources/open-source-hardware-definition/

- Open Source Hardware Basics:  
  https://certification.oshwa.org/basics.html

- OSHWA Certification - Documentation:  
  https://certification.oshwa.org/process/documentation.html

- OSHWA Certification FAQ:  
  https://certification.oshwa.org/basics/faq.html

### Ejemplos utilizados para la detección de tecnologías

- Buscar coincidencias de una lista de tecnologías mediante `re.findall()` y `re.IGNORECASE`:  
  https://stackoverflow.com/questions/51487012/how-to-print-matched-keywords-from-two-lists

- Buscar una lista de keywords dentro de textos mediante expresiones regulares:  
  https://stackoverflow.com/questions/66523520/python-how-to-use-list-of-keywords-to-search-for-a-string-in-a-text

- Construir una expresión regular con varias keywords mediante `|`:  
  https://stackoverflow.com/questions/15650244/using-regex-in-python-2-7-3-to-search-text-and-output-matches

- Python `re`:  
  https://docs.python.org/3/library/re.html
